In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import pandas as pd

In [ ]:
configs = json.loads((Path().resolve().parent.joinpath("config.json").read_text()))
whitelist = pd.read_csv(configs["whitelist"])[["uuid", "name"]]
df_raw = (pd.concat((pd.read_csv(f) for f in Path(configs["out"]).glob("*.csv")))
          .merge(whitelist, how="right", left_on="player_guid", right_on="uuid"))

In [ ]:
df = (df_raw[(df_raw["group"] == "custom") & (df_raw["stat"] == "play_time")] # filter to just play_time
      [["player_guid", "name", "timestamp", "value"]] # limit columns
      .groupby(["player_guid", "name"], as_index=False) # group by player
      .agg({"timestamp": "max", "value": "sum"}) # aggregate timestamp and value
      .sort_values("value", ascending=False) # sort by value desc
      .reset_index(drop=True) # reset index in sort order
   )
df

In [ ]:
# adding percent, cumulative percent, and grouping players after 90% of cumulative play time
df["pct"] = df["value"] / df["value"].sum()
df["pct_cum"] = df["pct"].cumsum()
df["group"] = df.apply(lambda x: "Other" if x["pct_cum"] >= 0.9 else x["name"], axis=1)
df["player_count"] = 1
df

In [ ]:
def clean_label(row) -> str:
   return (f"""
      {row["group"]}{f" ({int(row["player_count"])})" if row["group"] == "Other" else ""}
      {row["time"]} ({round(100 * row["pct"], 2)}%)
      {row["timestamp"]}
      """)

def clean_time(row) -> str:
   total_seconds = row["value"] / 20
   hours = 0 if total_seconds < 3600 else int(total_seconds / 3600)
   minutes = int((total_seconds % 3600) / 60)
   seconds = int(total_seconds % 60)
   return f"{hours:,}:{str(minutes).rjust(2, "0")}:{str(seconds).rjust(2, "0")}"

In [ ]:
view = pd.concat((
   df[df["group"] != "Other"]
   , df[df["group"] == "Other"]
      .groupby("group", as_index=False)
      .agg({"timestamp": "max", "value": "sum", "pct": "sum", "player_count": "sum"})
   ))
view["time"] = view.apply(clean_time, axis=1)
view["label"] = view.apply(clean_label, axis=1)
view

In [ ]:
plt.pie(view["value"], labels=view["label"].astype(str).tolist(), counterclock=False, startangle=90, radius=3)